# Backtest analysis

Backtest a bot, then look at where its PnL comes from: per-product PnL, fill quality, a
product's fills against the book, markouts (adverse selection), and a parameter grid.

Start Jupyter from the repo root: `uv run --group analysis jupyter lab notebooks/analysis.ipynb`.
Edit the settings in section 1 and run all cells. The same charts are available without Jupyter:
`uv run python -m arena.analysis results/<folder>`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from arena import analysis, backtest, grid

pd.set_option("display.width", 160)
# analysis.set_theme("dark")  # if Jupyter uses a dark theme

## 1. Backtest

`SETTINGS` takes any `arena.backtest` option, e.g. `"--match-trades", "worse"` or
`"--set", "PARAMS.KELP.skew=2"`.

In [ ]:
BOT = ROOT / "bots" / "template.py"
DAYS = ["1"]  # ROUND for all its days, or ROUND-DAY, e.g. ["1-0"]
SETTINGS = ["--data", "p3"]
OUT = ROOT / "results" / "notebook"

exit_code = backtest.main([str(BOT), *DAYS, *SETTINGS, "--out", str(OUT)])

## 2. PnL over the day

In [ ]:
DAY = analysis.days_in(OUT)[-1]
run = analysis.load_run(OUT, DAY)
analysis.plot_pnl(run)

## 3. Fills

- **took**: you crossed the spread and traded with a resting order.
- **passive**: one of your resting quotes was filled.
- **edge**: the average profit per unit, measured against the mid one tick before the fill.

An imbalance between `bought` and `sold` shows up as inventory drifting toward a limit.

In [ ]:
analysis.fill_summary(run)

## 4. One product

The full day shows inventory and PnL. A short window shows the mechanics: your quotes, which
of them got hit, and when you took liquidity.

In [ ]:
PRODUCT = run.products[0]
analysis.plot_product(run, PRODUCT)

In [ ]:
analysis.plot_product(run, PRODUCT, start=300_000, end=330_000)

## 5. Markouts

A markout is the mid price some ticks after a fill, minus the fill price (flipped for sells),
per unit. Flat lines mean your fills keep their value. Lines that fall as the horizon grows
mean the market moves against you after you trade, so you're being adversely selected. That's
the usual way market makers lose money.

In [ ]:
analysis.markouts(run)

In [ ]:
analysis.plot_markouts(run)

## 6. Parameter grid

Every combination is backtested on `DAYS` under each mode in `--modes`, in parallel. Don't
just take the best `pnl`. Prefer settings that also rank well on:

- **nbhd**: the mean PnL of a setting and its grid neighbours. A plateau is safer than a spike.
- **min day**: the setting's worst single day.
- **pnl under `worse`**: matching against market trades only when they print strictly through your price.

In [ ]:
GRID_OUT = ROOT / "results" / "notebook_grid.csv"
exit_code = grid.main([
    str(BOT), *DAYS, *SETTINGS,
    "--grid", "PARAMS.KELP.make_edge=0..3",
    "--grid", "PARAMS.KELP.take_edge=0..2",
    "--modes", "all,worse",
    "--out", str(GRID_OUT),
])

In [ ]:
grid_table = analysis.load_grid(GRID_OUT)
analysis.plot_grid(grid_table, "pnl_all")

In [ ]:
analysis.plot_grid(grid_table, "pnl_worse")